0. 환경 설정

In [1]:
!pip install -q pyarrow pandas numpy scikit-learn torch
!pip install -q matplotlib seaborn tqdm

1. 데이터 경로 설정

In [2]:
from google.colab import drive

drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/ch2025/'

Mounted at /content/drive


2. 데이터 로드

In [3]:
import os
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

FILE_MAP = {
    'ac_status': 'ch2025_mACStatus.parquet',
    'activity': 'ch2025_mActivity.parquet',
    'ambience': 'ch2025_mAmbience.parquet',
    'ble': 'ch2025_mBle.parquet',
    'gps': 'ch2025_mGps.parquet',
    'light': 'ch2025_mLight.parquet',
    'screen': 'ch2025_mScreenStatus.parquet',
    'usage_stats': 'ch2025_mUsageStats.parquet',
    'wifi': 'ch2025_mWifi.parquet',
    'hr': 'ch2025_wHr.parquet',
    'w_light': 'ch2025_wLight.parquet',
    'pedo': 'ch2025_wPedo.parquet',
}

raw = {}

for key, fname in FILE_MAP.items():
    path = os.path.join(
        DATA_DIR,
        'ch2025_data_items',
        fname
    )

    if os.path.exists(path):
        raw[key] = pd.read_parquet(path)

print(f'Loaded tables: {len(raw)}')

Loaded tables: 12


3. 라벨 데이터 로드

In [4]:
LABEL_COLS = ['Q1', 'Q2', 'Q3', 'S1', 'S2', 'S3', 'S4']

LABEL_PATH = os.path.join(DATA_DIR, 'ch2026_metrics_train.csv')

if not os.path.exists(LABEL_PATH):
    raise FileNotFoundError(f'Label file not found: {LABEL_PATH}')

labels_df = pd.read_csv(LABEL_PATH)

4. 피처 엔지니어링 (일별 집계)

*   1) 공통 유틸 함수

In [38]:
#타임스탬프 컬럼 자동 탐지
def get_timestamp_col(df):
    candidates = ['timestamp', 'time', 'datetime', 'ts', 'date']
    for c in df.columns:
        if any(k in c.lower() for k in candidates):
            return c
    # datetime dtype으로 탐지
    for c in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            return c
    return None

#object 센서용 유틸
def ensure_datetime(df, ts_col='timestamp'):
    df = df.copy()
    df[ts_col] = pd.to_datetime(df[ts_col], errors='coerce')
    df['date'] = pd.to_datetime(df[ts_col].dt.date)
    df['hour'] = df[ts_col].dt.hour
    return df

#list 센서용 유틸
def to_list(x):
    if x is None:
        return []
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, list):
        return x
    if isinstance(x, dict):
        return [x]
    return []



*   2) 기본 일별 집계 함수



In [39]:
def extract_daily_features(df, sensor_name, ts_col=None, subject_col='subject_id'):
    df = df.copy()

    # 타임스탬프 처리
    if ts_col is None:
        ts_col = get_timestamp_col(df)

    if ts_col and ts_col in df.columns:
        df[ts_col] = pd.to_datetime(df[ts_col], unit='ms', errors='coerce') \
                     if df[ts_col].dtype in ['int64','float64'] \
                     else pd.to_datetime(df[ts_col], errors='coerce')
        df['date'] = df[ts_col].dt.date
    elif 'date' not in df.columns:
        print(f'[{sensor_name}] 타임스탬프 컬럼을 찾지 못했습니다.')
        return None

    df['date'] = pd.to_datetime(df['date'])

    # 수치형 컬럼만 선택 (subject_id, date 제외)
    exclude = {subject_col, 'date', ts_col}
    num_cols = [c for c in df.select_dtypes(include=[np.number]).columns
                if c not in exclude]

    if not num_cols:
        print(f'[{sensor_name}] 수치형 컬럼이 없습니다.')
        return None

    # 하루 단위 집계
    grp = df.groupby([subject_col, 'date'])[num_cols]
    aggs = {
        'mean' : grp.mean(),
        'std'  : grp.std().fillna(0),
        'min'  : grp.min(),
        'max'  : grp.max(),
        'count': grp.count(),
    }

    # 멀티-집계 컬럼 병합
    result_parts = []
    for agg_name, agg_df in aggs.items():
        agg_df.columns = [f'{sensor_name}__{c}__{agg_name}' for c in agg_df.columns]
        result_parts.append(agg_df)

    result = pd.concat(result_parts, axis=1).reset_index()
    print(f'[{sensor_name}]: {result.shape[1]-2}개 피처, {result.shape[0]}행')
    return result


print('피처 엔지니어링 함수 정의 완료')

피처 엔지니어링 함수 정의 완료




*   3) 센서별 전용 피처 함수




In [40]:
# Screen 전용 수면 피처

def extract_screen_sleep_features(df, subject_col='subject_id'):
    df = ensure_datetime(df)
    rows = []

    for (subj, date), g in df.groupby([subject_col, 'date']):
        g = g.sort_values('timestamp').copy()

        # screen 사용 여부
        screen = pd.to_numeric(g['m_screen_use'], errors='coerce').fillna(0)
        hours = g['hour'].values
        times = g['timestamp'].values

        is_on = screen > 0

        # 시간대 마스크
        evening_mask = (hours >= 20) & (hours <= 23)
        late_evening_mask = (hours >= 22) & (hours <= 23)
        night_mask = (hours >= 0) & (hours <= 5)
        morning_mask = (hours >= 5) & (hours <= 10)

        # 마지막 screen on 시각
        on_times = pd.to_datetime(times[is_on.values])

        if len(on_times) > 0:
            last_on = max(on_times)
            last_on_hour = last_on.hour + last_on.minute / 60
            if last_on_hour < 6:
                last_on_hour += 24
        else:
            last_on_hour = 0

        # 첫 아침 screen on 시각
        morning_on_times = pd.to_datetime(times[(is_on.values) & morning_mask])
        if len(morning_on_times) > 0:
            first_morning_on = min(morning_on_times)
            first_morning_on_hour = first_morning_on.hour + first_morning_on.minute / 60
        else:
            first_morning_on_hour = 0

        feat = {
            'subject_id': subj,
            'date': date,

            # 하루 전체 screen 사용
            'screen_sleep__on_count_total': int(is_on.sum()),
            'screen_sleep__on_ratio_total': float(is_on.mean()) if len(is_on) else 0,

            # 취침 전 screen 사용
            'screen_sleep__on_count_20_23': int(is_on[evening_mask].sum()) if evening_mask.sum() else 0,
            'screen_sleep__on_ratio_20_23': float(is_on[evening_mask].mean()) if evening_mask.sum() else 0,

            'screen_sleep__on_count_22_23': int(is_on[late_evening_mask].sum()) if late_evening_mask.sum() else 0,
            'screen_sleep__on_ratio_22_23': float(is_on[late_evening_mask].mean()) if late_evening_mask.sum() else 0,

            # 새벽 screen 사용: WASO proxy
            'screen_sleep__on_count_00_05': int(is_on[night_mask].sum()) if night_mask.sum() else 0,
            'screen_sleep__on_ratio_00_05': float(is_on[night_mask].mean()) if night_mask.sum() else 0,

            # 시간 피처
            'screen_sleep__last_on_hour': last_on_hour,
            'screen_sleep__first_morning_on_hour': first_morning_on_hour,
        }

        rows.append(feat)

    result = pd.DataFrame(rows).fillna(0)
    print(f'[screen_sleep]: {result.shape[1]-2}개 피처')
    return result

In [41]:
# Light 전용 수면 피처

def extract_light_sleep_features(df, value_col='m_light', subject_col='subject_id'):
    df = ensure_datetime(df)
    df[value_col] = pd.to_numeric(df[value_col], errors='coerce').fillna(0)
    df[value_col] = df[value_col].clip(lower=0, upper=10000)

    rows = []

    for (subj, date), g in df.groupby([subject_col, 'date']):
        g = g.sort_values('timestamp').copy()

        light = g[value_col].values
        hours = g['hour'].values
        times = g['timestamp'].values

        evening_mask = (hours >= 20) & (hours <= 23)
        late_evening_mask = (hours >= 22) & (hours <= 23)
        night_mask = (hours >= 0) & (hours <= 5)

        # 어두운 환경 기준: lux 10 이하
        dark_mask = light <= 10

        # 마지막 밝은 환경 시각
        bright_mask = light > 10
        bright_times = pd.to_datetime(times[bright_mask])

        if len(bright_times) > 0:
            last_bright = max(bright_times)
            last_bright_hour = last_bright.hour + last_bright.minute / 60
            if last_bright_hour < 6:
                last_bright_hour += 24
        else:
            last_bright_hour = 0

        feat = {
            'subject_id': subj,
            'date': date,

            # 하루 전체 밝기
            'light_sleep__mean_total': np.mean(light) if len(light) else 0,
            'light_sleep__max_total': np.max(light) if len(light) else 0,
            'light_sleep__std_total': np.std(light) if len(light) else 0,

            # 저녁 밝기
            'light_sleep__mean_20_23': np.mean(light[evening_mask]) if evening_mask.sum() else 0,
            'light_sleep__max_20_23': np.max(light[evening_mask]) if evening_mask.sum() else 0,

            # 취침 직전 밝기
            'light_sleep__mean_22_23': np.mean(light[late_evening_mask]) if late_evening_mask.sum() else 0,
            'light_sleep__max_22_23': np.max(light[late_evening_mask]) if late_evening_mask.sum() else 0,

            # 새벽 밝기
            'light_sleep__mean_00_05': np.mean(light[night_mask]) if night_mask.sum() else 0,
            'light_sleep__max_00_05': np.max(light[night_mask]) if night_mask.sum() else 0,

            # 어두운 환경 비율
            'light_sleep__dark_ratio_total': np.mean(dark_mask) if len(dark_mask) else 0,
            'light_sleep__dark_ratio_22_23': np.mean(dark_mask[late_evening_mask]) if late_evening_mask.sum() else 0,
            'light_sleep__dark_ratio_00_05': np.mean(dark_mask[night_mask]) if night_mask.sum() else 0,

            # 마지막 밝은 환경 시각
            'light_sleep__last_bright_hour': last_bright_hour,
        }

        rows.append(feat)

    result = pd.DataFrame(rows).fillna(0)
    print(f'[light_sleep]: {result.shape[1]-2}개 피처')
    return result

In [42]:
# HR 기반 수면 구간 추정
def estimate_sleep_from_hr(df, subject_col='subject_id',
                            min_sleep_duration_min=60,
                            low_hr_window_min=15):
    """
    HR이 개인 임계값 아래로 sustained 하게 떨어지는 구간 = 수면 구간

    반환: 피험자×날짜별 추정 수면 시작/종료/길이 피처
    """
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', errors='coerce')
    df = df.explode('heart_rate')
    df['heart_rate'] = pd.to_numeric(df['heart_rate'], errors='coerce')
    df = df.dropna(subset=['heart_rate', 'timestamp'])
    df = df[df['heart_rate'].between(30, 220)]
    df = df.sort_values([subject_col, 'timestamp'])

    results = []

    for subj, subj_df in df.groupby(subject_col):
        # 개인 임계값: 전체 HR 하위 35% → "수면 HR" 기준
        sleep_threshold = subj_df['heart_rate'].quantile(0.35)

        # 날짜 기준: 전날 20시 ~ 당일 12시 (수면 윈도우)
        dates = subj_df['timestamp'].dt.date.unique()

        for date in dates:
            date = pd.Timestamp(date)
            window_start = date - pd.Timedelta(hours=4)   # 전날 20시
            window_end   = date + pd.Timedelta(hours=12)  # 당일 12시

            window = subj_df[
                (subj_df['timestamp'] >= window_start) &
                (subj_df['timestamp'] <= window_end)
            ].copy()

            if len(window) < 10:
                continue

            # 1분 단위 리샘플 후 rolling mean으로 노이즈 제거
            window = window.set_index('timestamp')['heart_rate']
            window_1min = window.resample('1min').mean().interpolate()

            # 임계값 아래 구간 탐지
            is_low = window_1min < sleep_threshold

            # 연속 구간 찾기
            sleep_onset = None
            sleep_offset = None
            max_duration = 0

            in_sleep = False
            seg_start = None

            for t, low in is_low.items():
                if low and not in_sleep:
                    in_sleep = True
                    seg_start = t
                elif not low and in_sleep:
                    duration = (t - seg_start).total_seconds() / 60
                    if duration > max_duration and duration >= min_sleep_duration_min:
                        max_duration = duration
                        sleep_onset  = seg_start
                        sleep_offset = t
                    in_sleep = False

            # 수면 구간 피처 계산
            row = {subject_col: subj, 'date': date}

            if sleep_onset and sleep_offset:
                sleep_hr = window_1min[sleep_onset:sleep_offset]
                pre_sleep_hr = window_1min[
                    max(window_1min.index[0], sleep_onset - pd.Timedelta(hours=1))
                    :sleep_onset
                ]

                row.update({
                    'hr__sleep_onset_hour':  sleep_onset.hour + sleep_onset.minute/60,
                    'hr__sleep_offset_hour': sleep_offset.hour + sleep_offset.minute/60,
                    'hr__est_tst_min':       max_duration,               # 추정 TST
                    'hr__sleep_hr_mean':     sleep_hr.mean(),
                    'hr__sleep_hr_std':      sleep_hr.std(),
                    'hr__sleep_hr_min':      sleep_hr.min(),              # 가장 깊은 수면 HR
                    # 취침 전 대비 수면 HR 감소폭 (클수록 회복 수면)
                    'hr__presleep_hr_drop':  (pre_sleep_hr.mean() - sleep_hr.mean()
                                              if len(pre_sleep_hr) > 0 else np.nan),
                    # 수면 중 HR 상승 이벤트 (각성 proxy → S4 WASO)
                    'hr__arousal_count':     int((sleep_hr > sleep_threshold).sum()),
                })
            else:
                # 수면 감지 실패 → NaN
                for col in ['hr__sleep_onset_hour','hr__sleep_offset_hour',
                            'hr__est_tst_min','hr__sleep_hr_mean','hr__sleep_hr_std',
                            'hr__sleep_hr_min','hr__presleep_hr_drop','hr__arousal_count']:
                    row[col] = np.nan

            results.append(row)

    result_df = pd.DataFrame(results)
    print(f'[hr_sleep]: {result_df.shape[1]-2}개 피처, {len(result_df)}행')
    return result_df


In [43]:
# WiFi 취침 시각 프록시
def extract_wifi_bedtime_proxy(df, subject_col='subject_id'):
    """
    야간 WiFi 스캔이 끊기는 시각 → 폰 내려놓은 시각 추정
    """
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', errors='coerce')
    df['date'] = df['timestamp'].dt.date
    df['date'] = pd.to_datetime(df['date'])
    df['hour'] = df['timestamp'].dt.hour

    results = []
    for (subj, date), grp in df.groupby([subject_col, 'date']):
        # 야간 스캔 (20시~다음날 6시)
        night = grp[grp['hour'].between(20, 23) | grp['hour'].between(0, 6)]
        night = night.sort_values('timestamp')

        row = {subject_col: subj, 'date': date}

        if len(night) >= 2:
            # 스캔 간격이 갑자기 길어지는 시점 = 폰 내려놓은 시각
            night = night.set_index('timestamp')
            gaps = night.index.to_series().diff().dt.total_seconds() / 60  # 분

            # 30분 이상 공백이 처음 생기는 시점
            long_gap = gaps[gaps > 30]
            if len(long_gap) > 0:
                gap_start = long_gap.index[0] - pd.Timedelta(minutes=gaps[long_gap.index[0]])
                row['wifi__phone_down_hour'] = gap_start.hour + gap_start.minute/60
            else:
                row['wifi__phone_down_hour'] = np.nan

            # 마지막 야간 스캔 시각
            last_scan = night.index[-1]
            row['wifi__last_night_scan_hour'] = last_scan.hour + last_scan.minute/60

            # 첫 아침 스캔 (5~9시)
            morning = grp[grp['hour'].between(5, 9)].sort_values('timestamp')
            row['wifi__first_morning_scan_hour'] = (
                morning['timestamp'].iloc[0].hour + morning['timestamp'].iloc[0].minute/60
                if len(morning) > 0 else np.nan
            )
        else:
            row.update({'wifi__phone_down_hour': np.nan,
                        'wifi__last_night_scan_hour': np.nan,
                        'wifi__first_morning_scan_hour': np.nan})

        results.append(row)

    result_df = pd.DataFrame(results)
    print(f'[wifi_bedtime]: {result_df.shape[1]-2}개 피처, {len(result_df)}행')
    return result_df


In [44]:
# Usage Stats 취침·기상 시각 프록시
def extract_usage_bedtime_proxy(df, subject_col='subject_id'):
    """
    마지막 앱 사용 시각 → 취침 시각
    첫 아침 앱 사용 → 기상 시각
    새벽 앱 사용 시간/앱 종류 → 수면 시작 지연(S3), 수면 중 각성(S4) proxy
    """
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df['date'] = pd.to_datetime(df['timestamp'].dt.date)
    df['hour'] = df['timestamp'].dt.hour

    social_keywords = [
        '카카오톡', 'kakao', 'instagram', '인스타',
        'facebook', 'messenger', 'telegram', 'line'
    ]

    media_keywords = [
        'youtube', '유튜브', 'netflix', '넷플릭스',
        'tiktok', '틱톡', 'wavve', 'watcha',
        '웹툰', 'naver', '티빙', 'tving'
    ]

    results = []

    for (subj, date), grp in df.groupby([subject_col, 'date']):
        grp = grp.sort_values('timestamp').copy()

        row = {
            subject_col: subj,
            'date': date
        }

        # 20~23시 앱 사용
        evening = grp[grp['hour'].between(20, 23)].sort_values('timestamp')
        if len(evening) > 0:
            last = evening['timestamp'].iloc[-1]
            row['usage__last_use_hour'] = last.hour + last.minute / 60
        else:
            row['usage__last_use_hour'] = np.nan

        # 22~23시 앱 사용 횟수
        row['usage__late_evening_count'] = grp['hour'].between(22, 23).sum()

        # 0~5시 앱 사용 횟수
        row['usage__night_count'] = grp['hour'].between(0, 5).sum()

        # 기존 수면 시간대 앱 사용 횟수
        row['usage__sleep_hour_count'] = grp['hour'].between(0, 6).sum()

        # 앱 사용 시간 기반 피처
        evening_total_time = 0.0
        late_evening_total_time = 0.0
        night_total_time = 0.0
        night_app_count = 0
        night_social_time = 0.0
        night_media_time = 0.0

        for _, r in grp.iterrows():
            hour = r['hour']
            apps = to_list(r['m_usage_stats'])

            for app in apps:
                if not isinstance(app, dict):
                    continue

                app_name = str(app.get('app_name', '')).strip().lower()

                try:
                    total_time = float(app.get('total_time', 0) or 0)
                except:
                    total_time = 0.0

                if total_time <= 0:
                    continue

                # 20~23시 전체 앱 사용 시간
                if 20 <= hour <= 23:
                    evening_total_time += total_time

                # 22~23시 앱 사용 시간
                if 22 <= hour <= 23:
                    late_evening_total_time += total_time

                # 0~5시 앱 사용 시간
                if 0 <= hour <= 5:
                    night_total_time += total_time
                    night_app_count += 1

                    if any(k in app_name for k in social_keywords):
                        night_social_time += total_time

                    if any(k in app_name for k in media_keywords):
                        night_media_time += total_time

        row['usage__evening_total_time'] = evening_total_time
        row['usage__late_evening_total_time'] = late_evening_total_time
        row['usage__night_total_time'] = night_total_time
        row['usage__night_app_count'] = night_app_count
        row['usage__night_social_time'] = night_social_time
        row['usage__night_media_time'] = night_media_time

        # 마지막 전체 앱 사용 시각
        if len(grp) > 0:
            last_active = grp['timestamp'].iloc[-1]
            last_active_hour = last_active.hour + last_active.minute / 60

            if last_active_hour < 6:
                last_active_hour += 24

            row['usage__last_active_hour'] = last_active_hour
        else:
            row['usage__last_active_hour'] = np.nan

        # 첫 아침 앱 사용 시각
        morning = grp[grp['hour'].between(5, 10)].sort_values('timestamp')
        if len(morning) > 0:
            first = morning['timestamp'].iloc[0]
            row['usage__first_morning_hour'] = first.hour + first.minute / 60
        else:
            row['usage__first_morning_hour'] = np.nan

        # 취침~기상 추정 공백 시간
        if (
            not pd.isna(row['usage__last_use_hour'])
            and
            not pd.isna(row['usage__first_morning_hour'])
        ):
            onset = row['usage__last_use_hour']
            offset = row['usage__first_morning_hour']

            gap = (
                offset - onset
                if offset > onset
                else 24 - onset + offset
            )

            row['usage__phone_off_duration_hr'] = gap
        else:
            row['usage__phone_off_duration_hr'] = np.nan

        row['usage__evening_total_hr'] = evening_total_time / 3600000
        row['usage__late_evening_total_hr'] = late_evening_total_time / 3600000
        row['usage__night_total_hr'] = night_total_time / 3600000
        row['usage__night_social_hr'] = night_social_time / 3600000
        row['usage__night_media_hr'] = night_media_time / 3600000

        results.append(row)

    result_df = pd.DataFrame(results).fillna(0)
    print(f'[usage_bedtime]: {result_df.shape[1]-2}개 피처, {len(result_df)}행')
    return result_df

In [45]:
# GPS 전용 피처

def extract_gps_special_features(df, subject_col='subject_id'):
    df = ensure_datetime(df)
    rows = []

    for (subj, date), g in df.groupby([subject_col, 'date']):
        speeds = []
        moving_times = []
        night_speed_flags = []

        for arr, ts, hour in zip(g['m_gps'].values, g['timestamp'].values, g['hour'].values):
            ts = pd.to_datetime(ts)

            for item in to_list(arr):
                if not isinstance(item, dict):
                    continue

                if item.get('speed') is not None:
                    spd = float(item.get('speed'))

                    if 0 <= spd <= 30:
                        speeds.append(spd)
                        is_moving = spd > 0.5

                        if is_moving:
                            moving_times.append(ts)

                        if 0 <= hour <= 5:
                            night_speed_flags.append(is_moving)

        speeds = np.array(speeds)

        if len(moving_times) > 0:
            last_move_time = max(moving_times)
            last_movement_hour = last_move_time.hour + last_move_time.minute / 60

            if last_movement_hour < 6:
                last_movement_hour += 24
        else:
            last_movement_hour = 0

        feat = {
            'subject_id': subj,
            'date': date,
            'gps__moving_ratio': np.mean(speeds > 0.5) if len(speeds) else 0,
            'gps__last_movement_hour': last_movement_hour,
            'gps__night_moving_ratio': np.mean(night_speed_flags) if len(night_speed_flags) else 0,
        }

        rows.append(feat)

    result = pd.DataFrame(rows).fillna(0)
    print(f'[gps_special]: {result.shape[1]-2}개 피처')
    return result

In [46]:
# Ambience 전용 피처

def extract_ambience_special_features(df, subject_col='subject_id'):
    df = ensure_datetime(df)

    target_labels = [
        'Speech',
        'Music',
        'Vehicle',
        'Silence',
        'Noise'
    ]

    rows = []

    for (subj, date), g in df.groupby([subject_col, 'date']):
        sums = {label: 0.0 for label in target_labels}
        top1_counts = {label: 0 for label in target_labels}

        total_events = 0

        evening_events = 0
        evening_speech = 0.0
        evening_music = 0.0
        evening_vehicle = 0.0
        evening_silence = 0.0
        evening_noise = 0.0

        for arr, hour in zip(g['m_ambience'].values, g['hour'].values):
            arr = to_list(arr)
            scores = {}

            for item in arr:
                item = to_list(item)

                if len(item) >= 2:
                    label = str(item[0])
                    try:
                        score = float(item[1])
                    except:
                        score = 0.0

                    scores[label] = score

                    if label in target_labels:
                        sums[label] += score

            if scores:
                total_events += 1
                top_label = max(scores, key=scores.get)

                if top_label in top1_counts:
                    top1_counts[top_label] += 1

                if 20 <= hour <= 23:
                    evening_events += 1
                    evening_speech += scores.get('Speech', 0.0)
                    evening_music += scores.get('Music', 0.0)
                    evening_vehicle += scores.get('Vehicle', 0.0)
                    evening_silence += scores.get('Silence', 0.0)
                    evening_noise += scores.get('Noise', 0.0)

        feat = {
            'subject_id': subj,
            'date': date,

            'ambience__speech__mean_score': sums['Speech'] / total_events if total_events else 0,
            'ambience__music__mean_score': sums['Music'] / total_events if total_events else 0,
            'ambience__vehicle__mean_score': sums['Vehicle'] / total_events if total_events else 0,
            'ambience__silence__mean_score': sums['Silence'] / total_events if total_events else 0,
            'ambience__noise__mean_score': sums['Noise'] / total_events if total_events else 0,

            'ambience__speech__top1_ratio': top1_counts['Speech'] / total_events if total_events else 0,
            'ambience__silence__top1_ratio': top1_counts['Silence'] / total_events if total_events else 0,

            'ambience__evening_speech_mean': evening_speech / evening_events if evening_events else 0,
            'ambience__evening_music_mean': evening_music / evening_events if evening_events else 0,
            'ambience__evening_vehicle_mean': evening_vehicle / evening_events if evening_events else 0,
            'ambience__evening_silence_mean': evening_silence / evening_events if evening_events else 0,
            'ambience__evening_noise_mean': evening_noise / evening_events if evening_events else 0,
        }

        rows.append(feat)

    result = pd.DataFrame(rows).fillna(0)
    print(f'[ambience_special]: {result.shape[1]-2}개 피처')
    return result

In [47]:
# BLE 전용 피처

def extract_ble_special_features(df, subject_col='subject_id'):
    df = ensure_datetime(df)
    rows = []

    for (subj, date), g in df.groupby([subject_col, 'date']):
        scan_device_counts = []
        rssis = []
        unique_addresses = set()
        device_classes = set()
        evening_counts = []
        night_counts = []

        for _, row in g.iterrows():
            devices = to_list(row['m_ble'])
            hour = row['hour']
            count = 0

            for dev in devices:
                if isinstance(dev, dict):
                    count += 1

                    addr = dev.get('address')
                    if addr:
                        unique_addresses.add(addr)

                    device_class = dev.get('device_class')
                    if device_class is not None:
                        device_classes.add(str(device_class))

                    rssi = dev.get('rssi')
                    if rssi is not None:
                        rssis.append(float(rssi))

            scan_device_counts.append(count)

            if 20 <= hour <= 23:
                evening_counts.append(count)

            if 0 <= hour <= 5:
                night_counts.append(count)

        scan_device_counts = np.array(scan_device_counts)
        rssis = np.array(rssis)

        feat = {
            'subject_id': subj,
            'date': date,
            'ble__scan_count': len(g),
            'ble__device_count_mean': np.mean(scan_device_counts) if len(scan_device_counts) else 0,
            'ble__device_count_std': np.std(scan_device_counts) if len(scan_device_counts) else 0,
            'ble__device_count_max': np.max(scan_device_counts) if len(scan_device_counts) else 0,
            'ble__unique_device_count': len(unique_addresses),
            'ble__unique_device_class_count': len(device_classes),
            'ble__rssi_mean': np.mean(rssis) if len(rssis) else 0,
            'ble__rssi_std': np.std(rssis) if len(rssis) else 0,
            'ble__rssi_max': np.max(rssis) if len(rssis) else 0,
            'ble__strong_signal_ratio': np.mean(rssis > -60) if len(rssis) else 0,
            'ble__evening_device_count_mean': np.mean(evening_counts) if len(evening_counts) else 0,
            'ble__night_device_count_mean': np.mean(night_counts) if len(night_counts) else 0,
        }

        rows.append(feat)

    result = pd.DataFrame(rows)
    print(f'[ble_special]: {result.shape[1]-2}개 피처')
    return result



*   4) 피처 생성 실행



In [48]:
daily_features = {}

# 기본 수치형 일별 집계 피처
for key, df in raw.items():
    feat = extract_daily_features(df, sensor_name=key)

    if feat is not None:
        daily_features[key] = feat

# object/list 센서 전용 피처
daily_features['gps_special'] = extract_gps_special_features(raw['gps'])
daily_features['ambience_special'] = extract_ambience_special_features(raw['ambience'])
daily_features['ble_special'] = extract_ble_special_features(raw['ble'])

# 수면 관련 전용 피처
daily_features['screen_sleep'] = extract_screen_sleep_features(raw['screen'])
daily_features['light_sleep'] = extract_light_sleep_features(raw['light'])
daily_features['usage_bedtime'] = extract_usage_bedtime_proxy(raw['usage_stats'])
daily_features['hr_sleep'] = estimate_sleep_from_hr(raw['hr'])

[ac_status]: 5개 피처, 700행
[activity]: 5개 피처, 700행
[ambience] 수치형 컬럼이 없습니다.
[ble] 수치형 컬럼이 없습니다.
[gps] 수치형 컬럼이 없습니다.
[light]: 5개 피처, 700행
[screen]: 5개 피처, 700행
[usage_stats] 수치형 컬럼이 없습니다.
[wifi] 수치형 컬럼이 없습니다.
[hr] 수치형 컬럼이 없습니다.
[w_light]: 5개 피처, 664행
[pedo]: 35개 피처, 653행
[gps_special]: 3개 피처
[ambience_special]: 12개 피처
[ble_special]: 12개 피처
[screen_sleep]: 10개 피처
[light_sleep]: 13개 피처
[usage_bedtime]: 18개 피처, 690행
[hr_sleep]: 8개 피처, 588행




*   5) feature_df 병합




In [16]:
from functools import reduce

feature_df = reduce(
    lambda l, r: pd.merge(l, r, on=['subject_id', 'date'], how='outer'),
    daily_features.values()
)

print(f'통합 피처: {feature_df.shape}')

통합 피처: (700, 138)




*   6) Rolling Feature 생성



In [17]:
feature_df = feature_df.sort_values(
    ['subject_id', 'date']
)

ROLL_COLS = [
    'gps__last_movement_hour',

    'screen_sleep__last_on_hour',
    'screen_sleep__on_ratio_22_23',
    'screen_sleep__on_ratio_00_05',

    'light_sleep__dark_ratio_22_23',
    'light_sleep__dark_ratio_00_05',

    'ambience__silence__top1_ratio'
]

# 실제 존재하는 컬럼만 사용
ROLL_COLS = [
    c for c in ROLL_COLS
    if c in feature_df.columns
]

for col in ROLL_COLS:

    # 최근 3일 평균
    feature_df[f'{col}_roll3'] = (
        feature_df
        .groupby('subject_id')[col]
        .transform(
            lambda x: x.rolling(
                window=3,
                min_periods=1
            ).mean()
        )
    )

    # 최근 7일 평균
    feature_df[f'{col}_roll7'] = (
        feature_df
        .groupby('subject_id')[col]
        .transform(
            lambda x: x.rolling(
                window=7,
                min_periods=1
            ).mean()
        )
    )

print(f'Rolling Feature 추가 완료')

Rolling Feature 추가 완료


In [18]:
#Q1 전용 Feature Engineering
feature_df['q1_bedtime_regularity'] = (
    feature_df['screen_sleep__last_on_hour_roll7']
    - feature_df['screen_sleep__last_on_hour']
).abs()

feature_df['q1_sleep_hygiene_score'] = (
    feature_df['light_sleep__dark_ratio_00_05']
    + feature_df['usage__phone_off_duration_hr']
    - feature_df['screen_sleep__on_ratio_00_05']
    - feature_df['usage__night_total_hr']
)

5. 개인화 편차 피처(Label Merge + Personalized Deviation Features)

In [19]:
labels_df['date'] = pd.to_datetime(labels_df['lifelog_date'])

full_df = pd.merge(
    feature_df,
    labels_df[['subject_id', 'date'] + LABEL_COLS],
    on=['subject_id', 'date'],
    how='inner'
)

print(f'레이블 병합 후: {full_df.shape}')

feat_cols = [
    c for c in full_df.columns
    if c not in ['subject_id', 'date'] + LABEL_COLS
]

# bool/object 컬럼을 숫자로 변환
for c in feat_cols:
    full_df[c] = full_df[c].replace({True: 1, False: 0})
    full_df[c] = pd.to_numeric(full_df[c], errors='coerce')

# 결측률 너무 높은 피처 제거
missing_ratio = full_df[feat_cols].isna().mean()
selected_feat_cols = missing_ratio[missing_ratio < 0.7].index.tolist()

print(f'기존 피처 수: {len(feat_cols)}')
print(f'결측률 기준 선별 후 피처 수: {len(selected_feat_cols)}')

# 결측값: subject별 median → 전체 median → 0
for c in selected_feat_cols:
    full_df[c] = full_df.groupby('subject_id')[c].transform(
        lambda x: x.fillna(x.median())
    )
    full_df[c] = full_df[c].fillna(full_df[c].median()).fillna(0)

ROLLING_WINDOW = 7
deviation_dfs = []

for subj_id, subj_df in full_df.groupby('subject_id'):
    subj_df = subj_df.sort_values('date').copy()

    rolling_mean = subj_df[selected_feat_cols].shift(1).rolling(
        window=ROLLING_WINDOW,
        min_periods=1
    ).mean()

    rolling_std = subj_df[selected_feat_cols].shift(1).rolling(
        window=ROLLING_WINDOW,
        min_periods=1
    ).std().fillna(1e-6)

    rolling_std = rolling_std.replace(0, 1e-6)

    deviation = (subj_df[selected_feat_cols] - rolling_mean) / rolling_std
    deviation = deviation.clip(-10, 10)
    deviation.columns = [f'dev__{c}' for c in selected_feat_cols]

    result = pd.concat([
        subj_df[['subject_id', 'date'] + LABEL_COLS].reset_index(drop=True),
        subj_df[selected_feat_cols].reset_index(drop=True),
        deviation.reset_index(drop=True)
    ], axis=1
    )

    deviation_dfs.append(result)

full_dev_df = pd.concat(deviation_dfs, ignore_index=True)
full_dev_df = full_dev_df.replace([np.inf, -np.inf], np.nan).fillna(0)

all_feat_cols = [
    c for c in full_dev_df.columns
    if c not in ['subject_id', 'date'] + LABEL_COLS
]

available_labels = LABEL_COLS

N_FEATURES = len(all_feat_cols)
N_LABELS = len(available_labels)

print(f'최종 데이터: {full_dev_df.shape}')
print(f'최종 입력 피처 수: {N_FEATURES}')

레이블 병합 후: (450, 161)
기존 피처 수: 152
결측률 기준 선별 후 피처 수: 144
최종 데이터: (450, 297)
최종 입력 피처 수: 288


6. 시퀀스 데이터셋 구성

In [20]:
import torch
from torch.utils.data import Dataset, DataLoader

SEQ_LEN = 7


class LifelogDataset(Dataset):
    def __init__(self, df, feat_cols, label_cols, seq_len=7, exclude_subjects=None):
        self.samples = []

        subjects = df["subject_id"].unique()

        if exclude_subjects:
            subjects = [s for s in subjects if s not in exclude_subjects]

        for subj in subjects:
            subj_df = df[df["subject_id"] == subj].sort_values("date")

            x_arr = subj_df[feat_cols].values.astype(np.float32)

            available = [c for c in label_cols if c in subj_df.columns]

            if available:
                y_arr = subj_df[available].values.astype(np.float32)
            else:
                y_arr = np.zeros(
                    (len(subj_df), len(label_cols)),
                    dtype=np.float32
                )

            x_arr = np.nan_to_num(
                x_arr,
                nan=0.0,
                posinf=0.0,
                neginf=0.0
            )

            x_arr = np.clip(
                x_arr,
                -10,
                10
            )

            for i in range(seq_len, len(subj_df)):
                x_seq = x_arr[i - seq_len:i]
                y_label = y_arr[i]

                if not np.any(np.isnan(y_label)):
                    self.samples.append(
                        (
                            torch.tensor(x_seq, dtype=torch.float32),
                            torch.tensor(y_label, dtype=torch.float32)
                        )
                    )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


raw_cols = [
    c for c in full_dev_df.columns
    if c not in ["subject_id", "date"] + LABEL_COLS
    and not c.startswith("dev__")
]

dev_cols = [
    c for c in full_dev_df.columns
    if c.startswith("dev__")
]

raw_cols = sorted(raw_cols)
dev_cols = sorted(dev_cols)

all_feat_cols = raw_cols + dev_cols

available_labels = [
    c for c in LABEL_COLS
    if c in full_dev_df.columns
]

N_FEATURES = len(all_feat_cols)
N_LABELS = len(available_labels)

print(f"Input features: {N_FEATURES}")
print(f"Labels: {N_LABELS}")

Input features: 288
Labels: 7


7. PTDT 모델 아키텍처

```
입력 (SEQ_LEN, N_FEATURES)
   ↓
Linear Projection → (SEQ_LEN, d_model)
   ↓
Positional Encoding
   ↓
Transformer Encoder (L layers)
   ↓
CLS Token Pooling
   ↓
7개 분류 헤드 (Q1, Q2, Q3, S1, S2, S3, S4)
```

In [21]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=365, dropout=0.1):
        super().__init__()

        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(0, max_len).unsqueeze(1).float()

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer(
            "pe",
            pe.unsqueeze(0)
        )

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class DeviationAwareAttention(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads,
        n_raw_feat,
        n_dev_feat
    ):
        super().__init__()

        self.raw_proj = nn.Linear(
            n_raw_feat,
            d_model
        )

        self.dev_proj = nn.Linear(
            n_dev_feat,
            d_model
        )

        self.fusion = nn.Linear(
            2 * d_model,
            d_model
        )

        self.attn = nn.MultiheadAttention(
            d_model,
            n_heads,
            batch_first=True
        )

        self.norm = nn.LayerNorm(d_model)

    def forward(self, raw_feat, dev_feat):

        raw_emb = F.gelu(
            self.raw_proj(raw_feat)
        )

        dev_emb = F.gelu(
            self.dev_proj(dev_feat)
        )

        fused = self.fusion(
            torch.cat(
                [raw_emb, dev_emb],
                dim=-1
            )
        )

        attn_out, _ = self.attn(
            dev_emb,
            raw_emb,
            raw_emb
        )

        return self.norm(
            fused + attn_out
        )


class PTDTransformer(nn.Module):

    def __init__(
        self,
        n_features,
        n_labels=7,
        d_model=64,
        n_heads=4,
        n_layers=2,
        d_ff=128,
        dropout=0.1,
        seq_len=7
    ):
        super().__init__()

        self.n_raw = n_features // 2
        self.n_dev = n_features - self.n_raw

        self.dev_attn = DeviationAwareAttention(
            d_model=d_model,
            n_heads=n_heads,
            n_raw_feat=self.n_raw,
            n_dev_feat=self.n_dev
        )

        self.cls_token = nn.Parameter(
            torch.randn(1, 1, d_model)
        )

        self.pos_enc = PositionalEncoding(
            d_model=d_model,
            max_len=seq_len + 1,
            dropout=dropout
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers
        )

        self.shared_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.LayerNorm(d_model)
        )

        self.task_heads = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Linear(d_model, 32),
                    nn.GELU(),
                    nn.Linear(32, 1)
                )
                for _ in range(n_labels)
            ]
        )

        self._init_weights()

    def _init_weights(self):

        for m in self.modules():

            if isinstance(m, nn.Linear):

                nn.init.xavier_uniform_(m.weight)

                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):

        batch_size = x.shape[0]

        raw = x[:, :, :self.n_raw]
        dev = x[:, :, self.n_raw:]

        emb = self.dev_attn(
            raw,
            dev
        )

        cls = self.cls_token.expand(
            batch_size,
            -1,
            -1
        )

        emb = torch.cat(
            [cls, emb],
            dim=1
        )

        emb = self.pos_enc(emb)

        enc = self.transformer(emb)

        cls_out = enc[:, 0]

        shared = self.shared_head(cls_out)

        logits = torch.cat(
            [
                head(shared)
                for head in self.task_heads
            ],
            dim=-1
        )

        return logits


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = PTDTransformer(
    n_features=N_FEATURES,
    n_labels=N_LABELS,
    d_model=64,
    n_heads=4,
    n_layers=2,
    d_ff=128,
    dropout=0.1,
    seq_len=SEQ_LEN
).to(device)

8. 학습 함수 정의

In [22]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score


def compute_metrics(y_true, y_pred_logit, threshold=0.5):
    y_prob = torch.sigmoid(torch.tensor(y_pred_logit)).numpy()
    y_pred = (y_prob >= threshold).astype(int)
    y_true = np.array(y_true)

    metrics = {}

    for i, label in enumerate(available_labels):
        y_label = y_true[:, i]
        pred_label = y_pred[:, i]
        prob_label = y_prob[:, i]

        mask = ~np.isnan(y_label)

        if mask.sum() == 0:
            continue

        metrics[label] = {
            "acc": accuracy_score(y_label[mask], pred_label[mask]),
            "f1": f1_score(y_label[mask], pred_label[mask], zero_division=0),
        }

        try:
            metrics[label]["auc"] = roc_auc_score(
                y_label[mask],
                prob_label[mask]
            )
        except ValueError:
            metrics[label]["auc"] = 0.5

    # Macro 평균
    if metrics:
        metrics["macro"] = {
            "acc": np.mean([v["acc"] for v in metrics.values()]),
            "f1": np.mean([v["f1"] for v in metrics.values()]),
            "auc": np.mean([v["auc"] for v in metrics.values()]),
        }

    return metrics


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits = model(x)
        loss = criterion(logits, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()

    all_logits = []
    all_labels = []

    for x, y in loader:
        x = x.to(device)

        logits = model(x)

        all_logits.append(logits.cpu().numpy())
        all_labels.append(y.numpy())

    all_logits = np.vstack(all_logits)
    all_labels = np.vstack(all_labels)

    return compute_metrics(
        all_labels,
        all_logits
    )

9. LOSO-CV 학습 (Leave-One-Subject-Out)

    10명 소규모 데이터에서 **과적합 방지** + **진정한 개인화 평가**를 위한 교차검증

In [32]:
# Reproducibility

import random

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [34]:
import copy

CONFIG = {
    "seq_len": SEQ_LEN,
    "d_model": 64,
    "n_heads": 4,
    "n_layers": 2,
    "d_ff": 128,
    "dropout": 0.1,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "epochs": 30,
    "batch_size": 16,
    "patience": 7,
}

all_subjects = sorted(full_dev_df["subject_id"].unique())
loso_results = {}

#데이터셋 분리: test subject 분리
for test_subj in all_subjects:
    test_df = full_dev_df[
        full_dev_df["subject_id"] == test_subj
    ]

    trainval_df = full_dev_df[
        full_dev_df["subject_id"] != test_subj
    ]

    candidate_subjects = [
        subj for subj in all_subjects
        if subj != test_subj
    ]

    # train subject 중 1명을 validation subject로 사용 + 순환방식
    test_idx = all_subjects.index(test_subj)
    val_subj = candidate_subjects[
        test_idx % len(candidate_subjects)
    ]

    train_df = trainval_df[
        trainval_df["subject_id"] != val_subj
    ]

    val_df = trainval_df[
        trainval_df["subject_id"] == val_subj
    ]

    train_ds = LifelogDataset(
        train_df,
        all_feat_cols,
        available_labels,
        CONFIG["seq_len"]
    )

    val_ds = LifelogDataset(
        val_df,
        all_feat_cols,
        available_labels,
        CONFIG["seq_len"]
    )

    test_ds = LifelogDataset(
        test_df,
        all_feat_cols,
        available_labels,
        CONFIG["seq_len"]
    )

    if len(train_ds) == 0 or len(val_ds) == 0 or len(test_ds) == 0:
        continue

    train_loader = DataLoader(
        train_ds,
        batch_size=CONFIG["batch_size"],
        shuffle=True,
        drop_last=False
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        drop_last=False
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        drop_last=False
    )

    # 모델 초기화
    model = PTDTransformer(
        n_features=N_FEATURES,
        n_labels=N_LABELS,
        d_model=CONFIG["d_model"],
        n_heads=CONFIG["n_heads"],
        n_layers=CONFIG["n_layers"],
        d_ff=CONFIG["d_ff"],
        dropout=CONFIG["dropout"],
        seq_len=CONFIG["seq_len"]
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"]
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=CONFIG["epochs"]
    )

    criterion = nn.BCEWithLogitsLoss()

# 학습 루프
    best_auc = 0
    best_model_state = None
    patience_count = 0

    history = {
        "train_loss": [],
        "val_auc": []
    }

    for epoch in range(1, CONFIG["epochs"] + 1):
        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device
        )

        val_metrics = evaluate(
            model,
            val_loader,
            device
        )

        scheduler.step()

        val_auc = val_metrics.get("macro", {}).get("auc", 0.5)

        history["train_loss"].append(train_loss)
        history["val_auc"].append(val_auc)

        if val_auc > best_auc:
            best_auc = val_auc
            best_model_state = copy.deepcopy(
                model.state_dict()
            )
            patience_count = 0
        else:
            patience_count += 1

        if patience_count >= CONFIG["patience"]:
            break

    if best_model_state is None:
        continue

  # 최적 모델로 최종 평가
    model.load_state_dict(best_model_state)

    final_metrics = evaluate(
        model,
        test_loader,
        device
    )

    loso_results[test_subj] = {
        "metrics": final_metrics,
        "history": history
    }

print("LOSO-CV completed")

LOSO-CV completed


10. 결과 분석 및 시각화

In [35]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

# ─── 전체 LOSO 결과 집계 ──────────────────────────────────
summary = {lbl: {'acc': [], 'f1': [], 'auc': []} for lbl in available_labels + ['macro']}

for subj, result in loso_results.items():
    for lbl, m in result['metrics'].items():
        if lbl in summary:
            summary[lbl]['acc'].append(m.get('acc', 0))
            summary[lbl]['f1'].append(m.get('f1',  0))
            summary[lbl]['auc'].append(m.get('auc', 0.5))

print('LOSO-CV 최종 결과 요약')
print(f'{"레이블":8s}  {"Acc":>8s}  {"F1":>8s}  {"AUC":>8s}')
print('-' * 42)
for lbl in available_labels + ['macro']:
    if summary[lbl]['auc']:
        acc_m = np.mean(summary[lbl]['acc'])
        f1_m  = np.mean(summary[lbl]['f1'])
        auc_m = np.mean(summary[lbl]['auc'])
        acc_s = np.std(summary[lbl]['acc'])
        f1_s  = np.std(summary[lbl]['f1'])
        auc_s = np.std(summary[lbl]['auc'])
        print(f'{lbl:8s}  '
              f'{acc_m:.3f}±{acc_s:.3f}  '
              f'{f1_m:.3f}±{f1_s:.3f}  '
              f'{auc_m:.3f}±{auc_s:.3f}')

LOSO-CV 최종 결과 요약
레이블            Acc        F1       AUC
------------------------------------------
Q1        0.447±0.148  0.410±0.278  0.457±0.125
Q2        0.601±0.137  0.661±0.246  0.519±0.129
Q3        0.610±0.129  0.750±0.100  0.526±0.137
S1        0.682±0.172  0.798±0.125  0.521±0.098
S2        0.638±0.219  0.756±0.173  0.434±0.170
S3        0.638±0.228  0.746±0.207  0.488±0.123
S4        0.545±0.188  0.648±0.200  0.508±0.142
macro     0.594±0.077  0.681±0.095  0.493±0.069


11. 모델 저장 및 예측 함수

In [25]:
print('전체 데이터로 최종 모델 재학습...')

full_ds = LifelogDataset(
    full_dev_df, all_feat_cols, available_labels, CONFIG['seq_len']
)
full_loader = DataLoader(full_ds, batch_size=CONFIG['batch_size'],
                         shuffle=True, drop_last=False)

final_model = PTDTransformer(
    n_features=N_FEATURES, n_labels=N_LABELS,
    d_model=CONFIG['d_model'], n_heads=CONFIG['n_heads'],
    n_layers=CONFIG['n_layers'], d_ff=CONFIG['d_ff'],
    dropout=0.05, seq_len=CONFIG['seq_len']  # dropout 낮춤
).to(device)

optimizer = torch.optim.AdamW(final_model.parameters(),
                               lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
criterion = nn.BCEWithLogitsLoss()

for epoch in range(1, CONFIG['epochs'] + 1):
    loss = train_one_epoch(final_model, full_loader, optimizer, criterion, device)
    if epoch % 10 == 0:
        print(f'  Epoch {epoch:3d}/{CONFIG["epochs"]} | Loss: {loss:.4f}')

# 저장
torch.save({
    'model_state': final_model.state_dict(),
    'config': CONFIG,
    'n_features': N_FEATURES,
    'n_labels': N_LABELS,
    'feat_cols': all_feat_cols,
    'label_cols': available_labels,
}, '/content/ptdt_model.pth')

print('모델 저장: /content/ptdt_model.pth')

전체 데이터로 최종 모델 재학습...
  Epoch  10/30 | Loss: 0.6227
  Epoch  20/30 | Loss: 0.4603
  Epoch  30/30 | Loss: 0.3433
모델 저장: /content/ptdt_model.pth


In [26]:
# 추론 함수 \

def predict_sleep_metrics(model, subject_history_df, feat_cols,
                          seq_len=SEQ_LEN, device=device):

    model.eval()

    df = subject_history_df.sort_values('date').tail(seq_len)

    if len(df) < seq_len:
        print(f'⚠️  {len(df)}일 데이터만 있습니다 (필요: {seq_len}일). 패딩 적용')
        pad_len = seq_len - len(df)
        pad_df = pd.DataFrame(0, index=range(pad_len), columns=df.columns)
        df = pd.concat([pad_df, df], ignore_index=True)

    # 피처 추출
    avail_cols = [c for c in feat_cols if c in df.columns]
    x = df[avail_cols].values.astype(np.float32)
    x = np.nan_to_num(x, nan=0.0)
    x = np.clip(x, -10, 10)

    # 누락 컬럼 처리
    if len(avail_cols) < len(feat_cols):
        full_x = np.zeros((seq_len, len(feat_cols)), dtype=np.float32)
        for i, col in enumerate(feat_cols):
            if col in avail_cols:
                full_x[:, i] = x[:, avail_cols.index(col)]
        x = full_x

    x_tensor = torch.tensor(x, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(x_tensor)
        probs = torch.sigmoid(logits).cpu().numpy()[0]

    result = {lbl: float(probs[i]) for i, lbl in enumerate(available_labels)}

    print('예측 결과 (확률값 > 0.5 → 긍정적 결과):')
    for lbl, prob in result.items():
        icon = '✅' if prob >= 0.5 else '⚠️ '
        bar = '█' * int(prob * 20) + '░' * (20 - int(prob * 20))
        print(f'  {icon} {lbl}: [{bar}] {prob:.3f}')

    return result
